In [9]:
import pandas as pd
data_dir = r"C:/Raj_Stuff/Coding/VScode/Workspaces/Astro-ML/Data/XMM-Gaia-Cross-Matched Data Separate"
# Load XMM and Gaia data from .parquet files
xmm_df = pd.read_parquet(f"{data_dir}/xmm_raw.parquet")
gaia_df = pd.read_parquet(f"{data_dir}/gaia_raw.parquet")
matched_df = pd.read_parquet(f"{data_dir}/cross_matched_raw.parquet")

In [10]:
from src.inference.encoder import xmm_encoder, gaia_encoder
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_dir = r"C:/Raj_Stuff/Coding/VScode/Workspaces/Astro-ML/Models"
xmm_encoder_path = f"{model_dir}/encoders/1e-3/simclr_xmm_3_frozen.pth"
gaia_encoder_path = f"{model_dir}/encoders/1e-3/simclr_gaia_3_frozen.pth"
xmm_enc=xmm_encoder(xmm_encoder_path)
gaia_enc=gaia_encoder(gaia_encoder_path)

In [11]:
gaia_cols = [
    'gaia_e_RA_ICRS', 'gaia_e_DE_ICRS',
    'gaia_pmRA', 'gaia_e_pmRA',
    'gaia_pmDE', 'gaia_e_pmDE',
    'gaia_Plx', 'gaia_e_Plx'
]

xmm_cols = [
    'SC_POSERR',
    'SC_DET_ML',
    'N_DETECTIONS',
    'CONFUSED',
    'HIGH_BACKGROUND',
    'No/Nx'
]

In [12]:
import numpy as np
import torch
from astropy.coordinates import SkyCoord
from astropy import units as u

def get_provisional_candidates(xmm_df, gaia_df, k_sigma=3.0):
    xmm_coords = SkyCoord(ra=xmm_df['SC_RA'].values*u.deg, 
                          dec=xmm_df['SC_DEC'].values*u.deg)
    gaia_coords = SkyCoord(ra=gaia_df['gaia_RA_ICRS'].values*u.deg, 
                           dec=gaia_df['gaia_DE_ICRS'].values*u.deg)
    
    # Broad search for all potential neighbors
    max_radius = k_sigma * xmm_df['SC_POSERR'].max() * u.arcsec
    idx_xmm, idx_gaia, sep2d, _ = gaia_coords.search_around_sky(xmm_coords, max_radius)
    
    # Filter by specific source error
    mask = sep2d.arcsec <= (k_sigma * xmm_df['SC_POSERR'].iloc[idx_xmm].values)
    
    return idx_xmm[mask], idx_gaia[mask], sep2d[mask].arcsec

In [13]:
import torch
import torch.nn.functional as F
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

def prepare_candidate_matrix(xmm_idx, xmm_df, gaia_df, ix, ig, seps):
    """
    Groups all Gaia candidates for a specific XMM source into tensors.
    """
    mask = (ix == xmm_idx)
    candidate_gaia_indices = ig[mask]
    candidate_separations = torch.tensor(seps[mask], dtype=torch.float32)
    
    # Extract features for the encoder
    xmm_feat = torch.tensor(xmm_df.iloc[xmm_idx][xmm_cols].values, dtype=torch.float32)
    gaia_feats = torch.tensor(gaia_df.iloc[candidate_gaia_indices][gaia_cols].values, dtype=torch.float32)
    poserr = xmm_df.iloc[xmm_idx]['SC_POSERR']
    
    return xmm_feat, gaia_feats, candidate_separations, poserr, candidate_gaia_indices

In [14]:
import torch.nn as nn
import torch.nn.functional as F

class XMMGaiaMatcher(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        # Bilinear bridge between your two pre-trained latent spaces
        self.W = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.02)
        self.null_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, z_xmm, z_gaia, sep_arcsec, xmm_poserr, alpha=1.0):
        """
        Returns a probability distribution over K candidates + 1 NULL option.
        """
        # 1. Compute scores: z_gaia(K, 64) @ W(64, 64) @ z_xmm(64, 1) -> (K,)
        semantic_score = z_gaia @ self.W @ z_xmm
        
        # 2. Add geometric prior: (distance / error)^2
        # Higher alpha = more trust in RA/Dec; Lower alpha = more trust in features
        geom_penalty = alpha * (sep_arcsec / xmm_poserr)**2
        logits = semantic_score - geom_penalty
        
        # 3. Append NULL bias (likelihood of no match)
        full_logits = torch.cat([logits, self.null_bias.view(1)])
        
        # 4. Probabilistic normalization
        probs = F.softmax(full_logits, dim=0)
        return probs

In [15]:
def train_and_evaluate(xmm_df, gaia_df, matched_df, ix, ig, seps, 
                       xmm_enc, gaia_enc, matcher, optimizer):
    xmm_enc.eval() # Keep pre-trained encoders fixed or set to train() to fine-tune
    gaia_enc.eval()
    matcher.train()
    
    total_loss = 0
    correct_matches = 0
    
    # We iterate through xmm_df (assuming it corresponds to matched_df)
    for i in range(len(xmm_df)):
        # Prepare data
        x_f, g_f, s, err, c_indices = prepare_candidate_matrix(i, xmm_df, gaia_df, ix, ig, seps)
        
        if len(c_indices) == 0: continue

        # Forward Pass
        z_xmm = xmm_enc(x_f)
        z_gaia = gaia_enc(g_f)
        probs = matcher(z_xmm, z_gaia, s, err)
        
        # Determine Ground Truth Index
        # Assuming matched_df has the correct 'gaia_row_index' for each XMM row
        true_gaia_idx = matched_df.iloc[i]['gaia_index_original'] 
        matching_indices = np.where(c_indices == true_gaia_idx)[0]
        
        if len(matching_indices) > 0:
            target_idx = torch.tensor(matching_indices[0])
        else:
            target_idx = torch.tensor(len(c_indices)) # Target is NULL

        # Loss (using NLL on log_softmax for stability)
        loss = F.nll_loss(torch.log(probs.unsqueeze(0)), target_idx.unsqueeze(0))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        if torch.argmax(probs) == target_idx:
            correct_matches += 1
        total_loss += loss.item()

    accuracy = correct_matches / len(xmm_df)
    return total_loss / len(xmm_df), accuracy

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as u

# --- 1. Device & Model Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Assuming xmm_enc and gaia_enc are already loaded and moved to device
xmm_enc.to(device)
gaia_enc.to(device)

# --- 2. Candidate Generation Logic ---

def get_provisional_candidates(xmm_df, gaia_df, k_sigma=3.0):
    xmm_coords = SkyCoord(ra=xmm_df['SC_RA'].values*u.deg, 
                          dec=xmm_df['SC_DEC'].values*u.deg)
    gaia_coords = SkyCoord(ra=gaia_df['gaia_RA_ICRS'].values*u.deg, 
                           dec=gaia_df['gaia_DE_ICRS'].values*u.deg)
    
    # Fast vectorized search
    max_radius = k_sigma * xmm_df['SC_POSERR'].max() * u.arcsec
    idx_xmm, idx_gaia, sep2d, _ = gaia_coords.search_around_sky(xmm_coords, max_radius)
    
    # Refine mask per-source error
    mask = sep2d.arcsec <= (k_sigma * xmm_df['SC_POSERR'].iloc[idx_xmm].values)
    return idx_xmm[mask], idx_gaia[mask], sep2d[mask].arcsec

def prepare_candidate_matrix(xmm_idx, xmm_df, gaia_df, ix, ig, seps, xmm_cols, gaia_cols):
    mask = (ix == xmm_idx)
    candidate_gaia_indices = ig[mask]
    
    # Features to Tensors
    xmm_feat = torch.tensor(xmm_df.iloc[xmm_idx][xmm_cols].values, dtype=torch.float32).to(device)
    poserr = torch.tensor(xmm_df.iloc[xmm_idx]['SC_POSERR'], dtype=torch.float32).to(device)
    
    if len(candidate_gaia_indices) > 0:
        gaia_feats = torch.tensor(gaia_df.iloc[candidate_gaia_indices][gaia_cols].values, dtype=torch.float32).to(device)
        candidate_seps = torch.tensor(seps[mask], dtype=torch.float32).to(device)
    else:
        # Placeholder for sources with no neighbors
        gaia_feats = torch.empty((0, len(gaia_cols))).to(device)
        candidate_seps = torch.empty((0,)).to(device)
    
    return xmm_feat, gaia_feats, candidate_seps, poserr, candidate_gaia_indices

# --- 3. The Matcher ---

class XMMGaiaMatcher(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.02)
        self.null_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, z_xmm, z_gaia, sep_arcsec, xmm_poserr, alpha=1.0):
        # Handle case with no candidates
        if z_gaia.shape[0] == 0:
            return torch.tensor([1.0], device=z_xmm.device) # 100% Null prob

        # Bilinear semantic score
        semantic_score = z_gaia @ self.W @ z_xmm
        # Gaussian prior penalty
        geom_penalty = alpha * (sep_arcsec / xmm_poserr)**2
        
        logits = semantic_score - geom_penalty
        full_logits = torch.cat([logits, self.null_bias.view(1)])
        
        return F.softmax(full_logits, dim=0)

# --- 4. Training & Inference ---

def train_one_epoch(xmm_df, gaia_df, matched_df, ix, ig, seps, 
                     xmm_enc, gaia_enc, matcher, optimizer, xmm_cols, gaia_cols):
    matcher.train()
    total_loss, correct = 0, 0
    
    for i in range(len(xmm_df)):
        x_f, g_f, s, err, c_indices = prepare_candidate_matrix(i, xmm_df, gaia_df, ix, ig, seps, xmm_cols, gaia_cols)
        
        # Pass through frozen encoders
        with torch.no_grad():
            z_xmm = xmm_enc(x_f)
            z_gaia = gaia_enc(g_f)

        probs = matcher(z_xmm, z_gaia, s, err)
        
        # Ground Truth mapping (Using your matched_df indices)
        # Note: adjust 'gaia_index_original' to the actual column name in your matched_df
        true_gaia_idx = matched_df.iloc[i]['gaia_index_original'] 
        matching_indices = np.where(c_indices == true_gaia_idx)[0]
        
        target_idx = torch.tensor(matching_indices[0] if len(matching_indices) > 0 else len(c_indices)).to(device)

        # Loss calculation (NLL loss requires log probabilities)
        loss = F.nll_loss(torch.log(probs + 1e-10).unsqueeze(0), target_idx.unsqueeze(0))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if torch.argmax(probs) == target_idx:
            correct += 1
        total_loss += loss.item()

    return total_loss / len(xmm_df), correct / len(xmm_df)

def predict_top_k(xmm_idx, k=3):
    """
    Utility to get readable predictions for a single XMM source.
    """
    x_f, g_f, s, err, c_indices = prepare_candidate_matrix(xmm_idx, xmm_df, gaia_df, ix, ig, seps, xmm_cols, gaia_cols)
    
    with torch.no_grad():
        probs = matcher(xmm_enc(x_f), gaia_enc(g_f), s, err)
    
    # Get top K from candidates (excluding the last element which is NULL)
    candidate_probs = probs[:-1]
    null_prob = probs[-1].item()
    
    # Clip K if fewer candidates than requested
    k = min(k, len(candidate_probs))
    top_p, top_idx = torch.topk(candidate_probs, k)
    
    results = [{"Gaia_Idx": c_indices[idx.item()], "Prob": p.item()} for p, idx in zip(top_p, top_idx)]
    results.append({"Label": "NULL", "Prob": null_prob})
    
    return results

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from astropy.coordinates import SkyCoord
from astropy import units as u

# --- 1. Setup & Config ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Columns as defined in your training
gaia_cols = ['gaia_e_RA_ICRS', 'gaia_e_DE_ICRS', 'gaia_pmRA', 'gaia_e_pmRA', 
             'gaia_pmDE', 'gaia_e_pmDE', 'gaia_Plx', 'gaia_e_Plx']
xmm_cols = ['SC_POSERR', 'SC_DET_ML', 'N_DETECTIONS', 'CONFUSED', 'HIGH_BACKGROUND', 'No/Nx']

# --- 2. Dataset & Collation ---

class XMMGaiaDataset(Dataset):
    def __init__(self, xmm_df, gaia_df, matched_df, ix, ig, seps):
        self.xmm_df = xmm_df
        self.gaia_df = gaia_df
        self.matched_df = matched_df
        self.ix, self.ig, self.seps = ix, ig, seps

    def __len__(self):
        return len(self.xmm_df)

    def __getitem__(self, idx):
        mask = (self.ix == idx)
        c_indices = self.ig[mask]
        
        x_feat = torch.tensor(self.xmm_df.iloc[idx][xmm_cols].values, dtype=torch.float32)
        poserr = torch.tensor(self.xmm_df.iloc[idx]['SC_POSERR'], dtype=torch.float32)
        
        # Features for all candidates
        g_feats = torch.tensor(self.gaia_df.iloc[c_indices][gaia_cols].values, dtype=torch.float32)
        s_vals = torch.tensor(self.seps[mask], dtype=torch.float32)
        
        # Ground Truth index mapping
        true_gaia_row = self.matched_df.iloc[idx]['gaia_index_original'] 
        truth_mask = (c_indices == true_gaia_row)
        target = torch.tensor(np.where(truth_mask)[0][0] if truth_mask.any() else len(c_indices), dtype=torch.long)

        return x_feat, g_feats, s_vals, poserr, target

def collate_fn(batch):
    """Pads variable number of Gaia candidates for batch processing."""
    x_feats, g_feats_list, s_vals_list, poserrs, targets = zip(*batch)
    
    # Pad Gaia features and separations to the max K in this batch
    max_k = max(g.size(0) for g in g_feats_list)
    padded_g = torch.stack([F.pad(g, (0, 0, 0, max_k - g.size(0))) for g in g_feats_list])
    padded_s = torch.stack([F.pad(s, (0, max_k - s.size(0)), value=1e6) for s in s_vals_list]) # High dist for padding
    
    # Create a mask so the model ignores padded candidates
    mask = torch.stack([torch.cat([torch.ones(g.size(0)), torch.zeros(max_k - g.size(0))]) for g in g_feats_list])
    
    return (torch.stack(x_feats), padded_g, padded_s, torch.stack(poserrs), torch.stack(targets), mask.to(device))

# --- 3. The Model ---

class XMMGaiaMatcher(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.02)
        self.null_bias = nn.Parameter(torch.tensor(0.0))

    def forward(self, z_xmm, z_gaia, sep_arcsec, xmm_poserr, mask, alpha=1.0):
        # z_xmm: (B, D), z_gaia: (B, K, D)
        # 1. Semantic Score (B, K)
        scores = torch.einsum('bkd,dd,bd->bk', z_gaia, self.W, z_xmm)
        
        # 2. Geometric Penalty (B, K)
        penalty = alpha * (sep_arcsec / xmm_poserr.unsqueeze(1))**2
        logits = scores - penalty
        
        # 3. Mask out padded candidates (set to very low value)
        logits = logits.masked_fill(mask == 0, -1e9)
        
        # 4. Append NULL bias to each batch
        full_logits = torch.cat([logits, self.null_bias.expand(logits.size(0), 1)], dim=1)
        return full_logits

# --- 4. Main Run Execution ---

# Pre-calculate candidates once
ix, ig, seps = get_provisional_candidates(xmm_df, gaia_df)

# Initialize
dataset = XMMGaiaDataset(xmm_df, gaia_df, matched_df, ix, ig, seps)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
matcher = XMMGaiaMatcher().to(device)
optimizer = torch.optim.Adam(matcher.parameters(), lr=1e-3)

# Training Loop
def train():
    matcher.train()
    for x_f, g_f, s, err, targets, mask in loader:
        x_f, g_f, s, err, targets = [item.to(device) for item in [x_f, g_f, s, err, targets]]
        
        # Get embeddings from your pre-loaded frozen encoders
        with torch.no_grad():
            z_xmm = xmm_enc(x_f)     # (B, 64)
            z_gaia = gaia_enc(g_f)   # (B, K, 64)

        logits = matcher(z_xmm, z_gaia, s, err, mask)
        loss = F.cross_entropy(logits, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch Loss: {loss.item():.4f}")

# Run one epoch
train()